# 07 · Delegated authority and sequential intervention
### BeliefLens · STAC AI Workshop · NYC · 21 October 2026

## Learning objectives
1. Connect explicit rules to accountable actions
2. Test permission revocation
3. Compare fixed gates with sequential decisions
4. Connect beliefs, consequences and information gathering

All companies, numbers, labels and market observations in these notebooks are synthetic teaching examples. Offline outputs come from explicit deterministic rules, not an LLM. The same JavaScript case engine runs in the browser and through this Python client. Only a cell explicitly using `live=True` calls Together AI. No financial action is executed.

**Requirements:** Python 3.10+, Node.js 20+, Jupyter. No model weights or API key are needed for the default exercises. Run cells in order.

In [ ]:
from pathlib import Path
import sys, json, math, random, statistics
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)
if root is None:
    raise RuntimeError("Extract the complete workshop ZIP and open Jupyter inside its folder.")
if str(root) not in sys.path: sys.path.insert(0, str(root))
from lab_client import run, compact, save
print("Shared engine ready:", compact(run()))

## The model proposes; the controller permits
Clear evidence and a simulated approval can permit RELEASE in this teaching workflow. Revocation must override an otherwise acceptable case. The browser’s approval checkbox is deliberately a simulation, not production identity or authorization.

In [ ]:
approved=run({"disruption":False,"approved":True})
revoked=run({"disruption":False,"approved":True,"revoked":True})
print(compact(approved));print(compact(revoked))
assert approved["gate"]=="RELEASE" and revoked["gate"]=="HOLD"
for rule in revoked["checks"]:
    print(rule["id"],rule["owner"],rule["status"],rule["reason"])

## A one-step partially observed decision problem
Hidden state: material credit harm or no harm. Belief $p$ is an assumed probability of harm. Actions: release, hold, or obtain a review signal and then choose. Release costs 100 if harm occurs; hold costs 8; review costs 2. The reviewer has sensitivity 0.8 and false-positive rate 0.1. These are explicit illustrative assumptions, not fitted BeliefLens reliability estimates.

Bayes’ rule updates the belief after the signal. Minimize expected loss after that update. This is a finite-horizon POMDP example with a static hidden state and one information-acquisition step.

In [ ]:
def terminal(p):
    losses={"release":100*p,"hold":8.0}
    return min(losses,key=losses.get),min(losses.values())
def decide(p):
    sensitivity,false_positive=0.8,0.1
    positive=p*sensitivity+(1-p)*false_positive
    p_pos=p*sensitivity/positive
    p_neg=p*(1-sensitivity)/(1-positive)
    after_pos,loss_pos=terminal(p_pos)
    after_neg,loss_neg=terminal(p_neg)
    review_loss=2+positive*loss_pos+(1-positive)*loss_neg
    direct,loss=terminal(p)
    return {"prior":p,"direct_action":direct,"direct_loss":loss,"review_loss":review_loss,
            "posterior_positive":p_pos,"posterior_negative":p_neg,
            "after_positive":after_pos,"after_negative":after_neg,
            "choice":"review" if review_loss<loss else direct}
for p in [0.01,0.05,0.08,0.2,0.6]:print(decide(p))
assert decide(0.08)["choice"]=="review"
assert decide(0.01)["choice"]=="release"
assert decide(0.6)["choice"]=="hold"

## What can be proved, and what must be tested?
A conditional software invariant might be: **if the current grant is revoked, the mediated execution path never releases.** A credible proof requires complete mediation, trusted permission state, atomic checks at commit, no bypass, and a correct model of the execution environment. These notebooks test examples; they do not prove a production invariant.

Reviewer sensitivity, evidence coverage, model reliability and financial loss assumptions require empirical testing or expert justification. Expected-loss optimization does not create authorization: an unauthorized action remains unavailable even when its assumed expected loss is small.

Agentic AI describes an acting system. Reinforcement learning is a way to learn policies from rewards. Both can use a sequential decision framework; an agent need not be trained with RL.

## Governance clinic deliverable
Complete this matrix for your own workflow:

| Rule | Node or dependency | Owner | Action if unresolved | Record |
|---|---|---|---|---|
| Eligible source | Evidence timestamps / permissions | Data steward | Hold and replace source | Source version and cutoff |
| Bound calculation | Evidence → tool | Credit analyst | Recompute or escalate | Formula and inputs |
| Sufficient support | Evidence / language → claim | Credit risk owner | Review significance | Claim, missing evidence, disposition |
| Authority | Current permission → action | Accountable approver | Block execution | Grant, version, decision time |

**Exit questions:** Who owns each rule? Which actions change future information? Where is the final permission check? What reliability claims need new labels? What would trigger revalidation?

## What you have demonstrated
- **Connect explicit rules to accountable actions:** Mapped checks to named roles and recorded dispositions
- **Test permission revocation:** Tested a revoked simulated grant
- **Compare fixed gates with sequential decisions:** Computed review versus immediate-action expected loss
- **Connect beliefs, consequences and information gathering:** Used Bayes updates under declared observation and loss assumptions

**Exit ticket:** State one result, the evidence supporting it, and one limit that prevents you from claiming more. Save a run JSON or your numerical results as the experiment record.